# 04. LTV по сегментам и unit-экономика

В ноутбуках 02 и 03 установлено, что клиенты с высоким первым чеком возвращаются заметно чаще. В этом ноутбуке статистический результат переводится в денежное выражение: оценивается LTV каждого клиента, проверяется, насколько различаются сегменты по совокупной выручке, и моделируется чувствительность окупаемости к стоимости привлечения.

Структура ноутбука:
1. Расчёт наблюдаемого LTV (совокупная выручка с клиента за период наблюдения), числа заказов и среднего чека.
2. Сравнение сегментов по LTV в денежном выражении.
3. Анализ концентрации выручки (правило Парето).
4. Простая модель unit-экономики и анализ чувствительности окупаемости к CAC.

Основной результат: сегмент с высоким первым чеком приносит за период наблюдения кратно большую выручку. Эффект объясняется не только разовым размером чека, но и большим числом повторных покупок и большей продолжительностью наблюдаемой жизни клиента.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_parquet('../data/clean.parquet')
customers = pd.read_parquet('../data/customers_labeled.parquet')
print(f'Транзакций: {len(df):,}')
print(f'Клиентов в сегментации: {len(customers):,}')

## Расчёт LTV

Под LTV в данном ноутбуке понимается наблюдаемая величина: совокупная выручка с клиента за весь доступный период наблюдения. Это не классический «lifetime» в смысле прогноза до бесконечности, а конечная оценка по двум годам данных.

Выбор подхода. Полноценная модель LTV (например, BG/NBD совместно с Gamma-Gamma) требует дополнительных допущений и существенно усложняет интерпретацию. Для бизнес-сравнения сегментов наблюдаемая величина даёт корректный ответ при сопоставимых временных горизонтах, и её точно можно объяснить продуктовой команде.

In [ ]:
ltv = (
    df.groupby('Customer ID')
    .agg(
        ltv_revenue=('Revenue', 'sum'),
        n_orders=('Invoice', 'nunique'),
        first_purchase=('InvoiceDate', 'min'),
        last_purchase=('InvoiceDate', 'max'),
    )
    .reset_index()
)
ltv['avg_order_value'] = ltv['ltv_revenue'] / ltv['n_orders']
ltv['lifetime_days'] = (ltv['last_purchase'] - ltv['first_purchase']).dt.days

ltv.head()

In [ ]:
# Присоединение информации о сегменте по размеру первого чека.
ltv = ltv.merge(
    customers[['Customer ID', 'check_segment', 'returned']],
    on='Customer ID',
    how='left',
)
ltv.head()

## Сравнение сегментов

Для оценки LTV приводятся одновременно среднее и медиана. Существенный разрыв между ними сигнализирует о тяжёлом правом хвосте; в этом случае медиана точнее описывает «типичного» клиента сегмента.

In [ ]:
segment_stats = (
    ltv.groupby('check_segment')
    .agg(
        n_customers=('Customer ID', 'count'),
        ltv_mean=('ltv_revenue', 'mean'),
        ltv_median=('ltv_revenue', 'median'),
        avg_orders=('n_orders', 'mean'),
        avg_aov=('avg_order_value', 'mean'),
        avg_lifetime_days=('lifetime_days', 'mean'),
    )
    .round(1)
)
segment_stats

Качественные наблюдения по таблице:
1. Средний LTV в сегменте «высокий первый чек» кратно превышает соответствующий показатель в сегменте «низкий первый чек» (различие измеряется не в процентах, а в разах).
2. Среднее число заказов также выше у сегмента с высоким чеком, что подтверждает: эффект не сводится к разовой крупной покупке.
3. Средняя продолжительность наблюдаемой «жизни» клиента в сегменте с высоким чеком существенно больше.

Полученные числа представляют независимое подтверждение результата t-теста из ноутбука 02, переведённое в финансовые показатели.

In [ ]:
# Распределение LTV в логарифмической шкале.
fig, ax = plt.subplots(figsize=(10, 5))
for seg, color in zip(
    ltv['check_segment'].dropna().unique(),
    ['#C44E52', '#55A868'],
):
    subset = ltv.loc[ltv['check_segment'] == seg, 'ltv_revenue']
    ax.hist(np.log10(subset[subset > 0]), bins=40, alpha=0.55, label=seg, color=color)

ax.set_title('Распределение LTV по сегментам (log10)')
ax.set_xlabel('log10(LTV в £)')
ax.set_ylabel('Количество клиентов')
ax.legend()
plt.tight_layout()
plt.savefig('../images/ltv_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

Логарифмическая шкала по горизонтальной оси использована из-за широкого диапазона значений LTV (от единиц до сотен тысяч фунтов). В линейной шкале гистограмма вырождается в одну палку у нуля; логарифмическое преобразование сохраняет интерпретируемость и делает форму распределений сопоставимой.

## Концентрация выручки: правило Парето

Доля выручки, приходящаяся на верхнюю часть базы по LTV, показывает, насколько концентрированно сосредоточена ценность. Высокая концентрация имеет прямые последствия для маркетингового планирования: предельная стоимость удержания клиента из верхнего децила существенно выше, чем из остальной базы.

In [ ]:
ltv_sorted = ltv.sort_values('ltv_revenue', ascending=False).reset_index(drop=True)
ltv_sorted['cum_revenue_pct'] = ltv_sorted['ltv_revenue'].cumsum() / ltv_sorted['ltv_revenue'].sum() * 100
ltv_sorted['cum_customers_pct'] = (ltv_sorted.index + 1) / len(ltv_sorted) * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ltv_sorted['cum_customers_pct'], ltv_sorted['cum_revenue_pct'], color='#4C72B0', linewidth=2)
ax.plot([0, 100], [0, 100], '--', color='gray', linewidth=1, label='линия равенства')
ax.axvline(x=20, color='red', linestyle=':', alpha=0.6)
ax.axhline(y=ltv_sorted.loc[int(len(ltv_sorted) * 0.20), 'cum_revenue_pct'], color='red', linestyle=':', alpha=0.6)

share_at_20 = ltv_sorted.loc[int(len(ltv_sorted) * 0.20), 'cum_revenue_pct']
ax.set_title(f'Концентрация выручки: топ-20% клиентов формируют {share_at_20:.0f}% выручки')
ax.set_xlabel('Доля клиентов, %')
ax.set_ylabel('Доля выручки, %')
ax.legend()
plt.tight_layout()
plt.savefig('../images/pareto_revenue.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Топ-20% клиентов приносят {share_at_20:.1f}% выручки')

Распределение выручки соответствует классическому правилу Парето. Практическое следствие: любая работа с retention должна учитывать асимметричную значимость клиентов. Потеря клиента из верхнего децила оказывает существенно большее влияние на выручку, чем потеря десяти клиентов из медианного диапазона.

## Модель unit-экономики

Поскольку датасет не содержит реальных данных по CAC и марже, в модели используются явно сформулированные допущения, соответствующие порядку величин для британской розницы.

Допущения:
- CAC = £20.
- Валовая маржа = 30% от выручки.
- Окупаемость определяется как отношение валовой маржи к CAC. Значение выше 1 означает, что сегмент компенсирует стоимость привлечения.

In [ ]:
CAC = 20.0  # £
GROSS_MARGIN = 0.30

unit_econ = (
    ltv.groupby('check_segment')
    .agg(ltv_mean=('ltv_revenue', 'mean'), ltv_median=('ltv_revenue', 'median'))
    .reset_index()
)
unit_econ['gross_profit_mean'] = unit_econ['ltv_mean'] * GROSS_MARGIN
unit_econ['gross_profit_median'] = unit_econ['ltv_median'] * GROSS_MARGIN
unit_econ['payback_x_mean'] = unit_econ['gross_profit_mean'] / CAC
unit_econ['payback_x_median'] = unit_econ['gross_profit_median'] / CAC
unit_econ.round(2)

Интерпретация колонок:
- payback_x_mean: кратность окупаемости среднего клиента сегмента. Значение 5.0 означает, что валовая маржа составляет 5x от CAC.
- payback_x_median: то же по медиане. Это более устойчивая оценка, поскольку среднее искажается тяжёлыми хвостами оптовых клиентов.

Содержательный вывод: даже по медианной оценке сегмент с высоким первым чеком окупается с существенным запасом, тогда как сегмент с низким первым чеком располагается ближе к границе безубыточности. Это означает, что любой рост стоимости привлечения (например, аукционная инфляция в performance-каналах) в первую очередь делает экономически неустойчивым именно низкий сегмент.

In [ ]:
# Анализ чувствительности окупаемости к CAC.
cac_grid = np.linspace(5, 100, 50)
fig, ax = plt.subplots(figsize=(10, 5))

for seg, color in zip(unit_econ['check_segment'], ['#C44E52', '#55A868']):
    median_ltv = unit_econ.loc[unit_econ['check_segment'] == seg, 'ltv_median'].iloc[0]
    payback = (median_ltv * GROSS_MARGIN) / cac_grid
    ax.plot(cac_grid, payback, label=seg, color=color, linewidth=2)

ax.axhline(y=1, color='black', linestyle='--', alpha=0.6, label='граница окупаемости (payback = 1)')
ax.set_xlabel('CAC, £')
ax.set_ylabel('Payback (валовая маржа / CAC), по медиане')
ax.set_title('Чувствительность окупаемости к стоимости привлечения')
ax.legend()
plt.tight_layout()
plt.savefig('../images/cac_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show()

График позволяет определить пороговое значение CAC, при котором каждый из сегментов теряет окупаемость. Если в реальности фиксируется приближение CAC к этому порогу для низкого сегмента, разумные ответы лежат в двух плоскостях: ограничение трафика на условия, генерирующие низкочековых клиентов, либо целевое повышение среднего чека первой покупки через продуктовые механики (бандлы, пороги бесплатной доставки, минимальная сумма заказа для промокодов).

## Сохранение результатов

In [ ]:
ltv.to_parquet('../data/ltv.parquet', index=False)
print('Сохранено: data/ltv.parquet')

## Резюме

1. Сегмент с высоким первым чеком приносит за период наблюдения кратно большую выручку. Эффект складывается из трёх компонент: более высокого AOV, большего числа повторных покупок и большей продолжительности наблюдаемого взаимодействия.
2. Распределение выручки в базе соответствует правилу Парето; клиенты верхнего децила имеют непропорционально высокий вес для совокупной экономики.
3. По упрощённой модели unit-экономики высокий сегмент окупается с существенным запасом, а низкий находится вблизи границы безубыточности. Это уточняет рекомендации ноутбука 03: триггерные кампании в низком сегменте обоснованы, но цена ошибки в этом сегменте выше, что повышает значимость аккуратного дизайна A/B-теста (ноутбук 05).